# Aula 5 — O fechamento de abril

**Julho.** A lista de julho chegou com **580 pessoas em alerta**. Antes de distribuí-la para o time, a Marina quer saber se dá para confiar nela: em abril, a lista indicou **581** pessoas em alerta e, no fechamento, **só 46** pediram demissão.

O sistema não responde isso — o monitor de julho só olha a entrada, e julho está verde. A resposta está no fechamento de abril, e é este notebook que olha para ele.

Este notebook tem duas partes, feitas em momentos diferentes da aula:

- **Parte 1** — Passo 1 do roteiro. A leitura de sempre: como a v3 se saiu em abril, e o que mudou na relação entre as features e a saída?
- **Parte 2** — Passo 3 do roteiro, depois da planilha do RH. A leitura que faltou: o que *nós* mudamos?

**O código já está pronto.** Rode célula a célula, da raiz do projeto, com o MLflow e o Postgres no ar. O exercício não é escrever o código: é ler cada resultado e responder a pergunta da seção antes de seguir. As perguntas da Parte 3, no fim, são a entrega.

Este notebook **só lê**: não grava no banco, não registra nada no MLflow e não altera nenhum arquivo do projeto.

## 0. Preparar

A célula abaixo importa o `config` e o `app.banco` do projeto e carrega:

1. `abril`: o `lote_aula4_abr` pontuado (use `banco.carregar_lote`), juntado com `dados/gabarito_aula4_abr.csv`;
2. `marco`: o `lote_aula3_mar_corrigido` pontuado, juntado com `dados/gabarito_aula3_mar.csv` — o último mês rotulado **antes** de abril;
3. uma coluna `faixa` nos dois (`alerta` a partir de `config.FAIXA_ALERTA`, `atencao` a partir de `config.FAIXA_ATENCAO`, senão `ok`).

Repare na taxa de saída de cada mês.

In [1]:
import sys
sys.path.insert(0, ".")
import warnings
warnings.filterwarnings("ignore")

import mlflow
import numpy as np
import pandas as pd

import config
from app import banco

ALVO = config.COLUNA_ALVO


def com_gabarito(lote, arquivo):
    df = banco.carregar_lote(lote)
    gab = pd.read_csv(config.PASTA_DADOS / arquivo, sep=";")
    gab[config.COLUNA_ID] = gab[config.COLUNA_ID].astype(str)
    df = df.merge(gab, on=config.COLUNA_ID)
    p = df["probabilidade"]
    df["faixa"] = np.select([p >= config.FAIXA_ALERTA, p >= config.FAIXA_ATENCAO],
                            ["alerta", "atencao"], "ok")
    return df


abril = com_gabarito("lote_aula4_abr", "gabarito_aula4_abr.csv")
marco = com_gabarito("lote_aula3_mar_corrigido", "gabarito_aula3_mar.csv")
print(f"marco  {len(marco)} linhas | saida {marco[ALVO].mean():.3f} | modelo v{marco['versao_modelo'].iloc[0]}")
print(f"abril  {len(abril)} linhas | saida {abril[ALVO].mean():.3f} | modelo v{abril['versao_modelo'].iloc[0]}")

marco  1400 linhas | saida 0.442 | modelo v2
abril  1400 linhas | saida 0.256 | modelo v3


---
# Parte 1 — A leitura de sempre

## 1.1 Como a v3 se saiu em abril

A Marina mediu à mão: 581 em alerta, 46 saíram. Aqui está o mesmo fechamento medido com as métricas das aulas anteriores. Cada linha compara o que a lista disse (a coluna `classe` do lote: 1 = “vai sair”) com o que aconteceu (o gabarito). Março, respondido pela v2, fica ao lado como referência.

Como ler cada linha:

| linha | pergunta que ela responde |
|---|---|
| **% marcados como “vai sair”** | quanto da lista o modelo pôs como risco |
| **% que saiu de verdade** | quanto saiu no fechamento |
| **dos marcados, quantos saíram** (*precision*) | quando a lista diz “vai sair”, ela acerta? |
| **dos que saíram, quantos estavam marcados** (*recall*) | a lista pegou quem saiu? |
| **F1** | uma nota só, que equilibra as duas linhas acima — a métrica do portão |
| **acerto geral** (*accuracy*) | em quantas pessoas a lista acertou, contando “sai” e “fica” |

**Perguntas:** qual das duas linhas do meio — *precision* ou *recall* — mudou de março para abril? Em palavras: a lista de abril deixou de pegar quem sai, ou marcou gente que não saiu?

In [2]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def desempenho(df):
    y, c = df[ALVO], df["classe"]
    return {
        "% marcados como 'vai sair'": c.mean(),
        "% que saiu de verdade": y.mean(),
        "dos marcados, quantos sairam (precision)": precision_score(y, c),
        "dos que sairam, quantos estavam marcados (recall)": recall_score(y, c),
        "F1 (a metrica do portao)": f1_score(y, c),
        "acerto geral (accuracy)": accuracy_score(y, c),
    }


alerta_abr = abril[abril["faixa"] == "alerta"]
print(f"abril: {len(alerta_abr)} em alerta, {int(alerta_abr[ALVO].sum())} sairam")
pd.DataFrame({"marco (v2)": desempenho(marco), "abril (v3)": desempenho(abril)}).round(3)

abril: 581 em alerta, 46 sairam


,marco (v2),abril (v3)
% marcados como 'vai sair',0.527,0.698
% que saiu de verdade,0.442,0.256
"dos marcados, quantos sairam (precision)",0.539,0.232
"dos que sairam, quantos estavam marcados (recall)",0.643,0.634
F1 (a metrica do portao),0.587,0.340
acerto geral (accuracy),0.599,0.371


## 1.2 Quem saiu, em cada faixa da lista

A lista separa as pessoas em três faixas: **alerta** (probabilidade a partir de 60%), **atenção** (50% a 60%) e **ok** (abaixo de 50%). Se a lista está certa, a faixa de alerta é a que mais sai e a ok, a que menos sai.

A tabela mostra, para cada faixa e cada mês, quantas pessoas estavam nela e qual porcentagem delas pediu demissão.

**Pergunta:** em março, a ordem das faixas fazia sentido (alerta saindo mais, ok saindo menos)? E em abril?

In [3]:
ordem = ["alerta", "atencao", "ok"]
pd.DataFrame({
    "pessoas em marco": marco["faixa"].value_counts(),
    "% que saiu em marco": marco.groupby("faixa")[ALVO].mean() * 100,
    "pessoas em abril": abril["faixa"].value_counts(),
    "% que saiu em abril": abril.groupby("faixa")[ALVO].mean() * 100,
}).reindex(ordem).round(1)

,pessoas em marco,% que saiu em marco,pessoas em abril,% que saiu em abril
faixa,,,,
alerta,165,57.0,581,7.9
atencao,573,53.1,396,45.7
ok,662,33.4,423,31.0


## 1.3 A relação de cada feature com a saída

**Isto não é a importância das features para o modelo.** É uma pergunta sobre o dado real: *quem tem mais desta feature sai mais ou sai menos?*

Para cada feature que a v3 lê:

1. As pessoas são divididas em duas metades pela mediana da feature em março: quem tem **menos** e quem tem **mais**.
2. Em cada mês, calcula-se a porcentagem de saída de cada metade.
3. A **diferença** é `% de saída de quem tem mais − % de saída de quem tem menos`.

Como ler a diferença:

- **negativa** — quem tem mais sai **menos**. A feature “protege”.
- **perto de zero** — tanto faz ter mais ou menos.
- **positiva** — quem tem mais sai **mais**. A feature indica risco.

Exemplo: se em março a diferença de `mov_sal_valor_total` é −17,3, quem teve mais movimentação salarial saiu 17,3 pontos percentuais **menos** do que quem teve pouca.

Features em que uma das metades tem menos de 30 pessoas ficam de fora (as colunas quase todas zeradas). A tabela vem ordenada pelas que mais mudaram de um mês para o outro.

**Pergunta:** alguma feature trocou de sinal entre março e abril? O que isso significaria, se fosse verdade?

In [4]:
mlflow.set_tracking_uri(config.URI_TRACKING)
v3 = mlflow.sklearn.load_model(f"models:/{config.NOME_MODELO}/3")
features_v3 = list(v3.feature_names_in_)


def saida_por_metade(df, coluna, corte):
    """% de saida de quem tem menos e de quem tem mais (em relacao ao corte)."""
    mais = df[coluna] > corte
    if mais.sum() < 30 or (~mais).sum() < 30:
        return np.nan, np.nan
    return df.loc[~mais, ALVO].mean() * 100, df.loc[mais, ALVO].mean() * 100


def diferenca(df, coluna, corte):
    menos, mais = saida_por_metade(df, coluna, corte)
    return mais - menos


linhas = []
for coluna in features_v3:
    corte = marco[coluna].median()
    m_menos, m_mais = saida_por_metade(marco, coluna, corte)
    a_menos, a_mais = saida_por_metade(abril, coluna, corte)
    linhas.append({"feature": coluna,
                   "marco: % saida, quem tem menos": m_menos,
                   "marco: % saida, quem tem mais": m_mais,
                   "marco: diferenca": m_mais - m_menos,
                   "abril: % saida, quem tem menos": a_menos,
                   "abril: % saida, quem tem mais": a_mais,
                   "abril: diferenca": a_mais - a_menos})
relacao = pd.DataFrame(linhas).dropna().set_index("feature")
ordem_mudanca = (relacao["abril: diferenca"] - relacao["marco: diferenca"]).abs().sort_values(ascending=False).index
relacao.loc[ordem_mudanca].round(1)

,"marco: % saida, quem tem menos","marco: % saida, quem tem mais",marco: diferenca,"abril: % saida, quem tem menos","abril: % saida, quem tem mais",abril: diferenca
feature,,,,,,
mov_sal_valor_total,52.8,35.5,-17.3,20.9,30.5,9.6
mov_sal_eventos,53.7,32.4,-21.3,23.2,28.6,5.4
mov_sal_perc_medio,52.8,35.5,-17.3,22.2,28.7,6.5
acidentes_eventos,46.1,32.7,-13.5,24.2,33.7,9.4
hi_eventos,50.2,37.9,-12.3,21.6,28.9,7.3
hi_minutos_irregulares,49.9,38.6,-11.3,24.0,27.1,3.1


## 1.4 A conclusão de sempre

Antes de ler a resposta abaixo, escreva a sua em duas frases.

> *A leitura de sempre:* a relação de `mov_sal_*` com a saída mudou de sinal. Em março, mais movimentação salarial protegia; em abril, deixou de proteger. **O conceito da feature mudou** — a mesma coluna, com os mesmos valores (o monitor de entrada não viu nada), passou a significar outra coisa.

Com essa conclusão, a resposta de sempre é a da Aula 4: tirar do modelo a coluna que mudou de significado — e, de quebra, a `hi_minutos_extras`, que está zerada desde o treino — e retreinar incorporando abril. **Volte ao roteiro, Passo 2.**

---
# Parte 2 — A leitura que faltou

*Rode no Passo 3 do roteiro, com a planilha da Marina em `dados/intervencoes_aula4_abr.csv`.*

## 2.1 Quem recebeu o reajuste?

A planilha é lida e cada pessoa de `abril` ganha a coluna `reajuste` (sim ou não). Depois, o cruzamento com a faixa da lista. Lembre: a faixa de alerta começa em probabilidade **0,60**.

**Pergunta:** a diretoria escolheu quem receberia o reajuste com algum critério próprio, ou usou a nossa lista?

In [5]:
reajuste = pd.read_csv(config.PASTA_DADOS / "intervencoes_aula4_abr.csv", sep=";")
reajuste[config.COLUNA_ID] = reajuste[config.COLUNA_ID].astype(str)
abril["reajuste"] = abril[config.COLUNA_ID].isin(reajuste[config.COLUNA_ID])

print(f"{len(reajuste)} reajustes; menor probabilidade da v3 entre quem recebeu: "
      f"{abril.loc[abril['reajuste'], 'probabilidade'].min():.3f}")
pd.crosstab(abril["faixa"], abril["reajuste"]).reindex(ordem)

581 reajustes; menor probabilidade da v3 entre quem recebeu: 0.600


reajuste,False,True
faixa,,
alerta,0,581
atencao,396,0
ok,423,0


## 2.2 Quem saiu em abril?

Agora olhe para quem pediu demissão. Para cada faixa: quantas pessoas, quantas receberam o reajuste, quantas saíram, a porcentagem que saiu — e, para comparar, a porcentagem que saiu da mesma faixa em março, quando ninguém recebeu nada.

**Pergunta:** a faixa que recebeu o reajuste se comportou como em março? E as que não receberam? Das 358 pessoas que saíram em abril, de onde veio a maioria?

In [6]:
quem_saiu = pd.DataFrame({
    "pessoas": abril["faixa"].value_counts(),
    "receberam reajuste": abril.groupby("faixa")["reajuste"].sum(),
    "sairam": abril.groupby("faixa")[ALVO].sum(),
    "% que saiu em abril": abril.groupby("faixa")[ALVO].mean() * 100,
    "% que saiu em marco": marco.groupby("faixa")[ALVO].mean() * 100,
}).reindex(ordem)
quem_saiu["% das saidas de abril"] = quem_saiu["sairam"] / quem_saiu["sairam"].sum() * 100
print(f"sairam em abril: {int(quem_saiu['sairam'].sum())} pessoas")
quem_saiu.round(1)

sairam em abril: 358 pessoas


,pessoas,receberam reajuste,sairam,% que saiu em abril,% que saiu em marco,% das saidas de abril
faixa,,,,,,
alerta,581,581,46,7.9,57.0,12.8
atencao,396,0,181,45.7,53.1,50.6
ok,423,0,131,31.0,33.4,36.6


**O que a tabela diz:**

- A faixa de **alerta**, que em março saía 57%, saiu **8%** em abril. Foi a única que recebeu o reajuste — e foi a única que mudou.
- **Atenção** e **ok**, que não receberam nada, saíram praticamente como em março (53% → 46%, 33% → 31%).
- Das 358 saídas de abril, **312 vieram de fora do alerta**: gente que a lista pôs em atenção ou ok e que ninguém tocou.

A lista não errou quem tinha risco: o reajuste segurou essas pessoas. A conta da Marina (“581 indicados, só 46 saíram”) não mede a lista — mede a lista **mais** o reajuste.

## 2.3 O que a v4 aprendeu com isso?

A v4 é o retreino do Passo 2: treinada com 70% de abril, ela viu essas 581 pessoas com o rótulo “ficou”. O portão a reprovou, mas ela existe no registry.

A célula compara a v3 e a v4 em dois momentos:

1. **Os 581 do alerta de abril**: que probabilidade média cada versão dá a eles, e quantos cada uma marca como “vai sair”.
2. **A lista de julho**: quantos cada versão põe em alerta, e para onde a v4 manda os 580 que a v3 põe em alerta.

**Pergunta:** em uma frase, o que a v4 aprendeu sobre quem está na faixa de alerta?

In [7]:
v4 = mlflow.sklearn.load_model(f"models:/{config.NOME_MODELO}/4")
julho = pd.read_csv(config.PASTA_DADOS / "lote_aula5_jul.csv", sep=";")


def prob(modelo, df):
    return modelo.predict_proba(df[list(modelo.feature_names_in_)])[:, 1]


def faixa_de(p):
    return pd.Series(np.select([p >= config.FAIXA_ALERTA, p >= config.FAIXA_ATENCAO],
                               ["alerta", "atencao"], "ok"))


alerta_abril = abril[abril["faixa"] == "alerta"]
print("os 581 do alerta de abril")
print(pd.DataFrame({
    "probabilidade media": [prob(v3, alerta_abril).mean(), prob(v4, alerta_abril).mean()],
    "marcados como 'vai sair'": [(prob(v3, alerta_abril) >= 0.5).sum(), (prob(v4, alerta_abril) >= 0.5).sum()],
}, index=["v3", "v4"]).round(3))

p3, p4 = prob(v3, julho), prob(v4, julho)
print("\na lista de julho")
print(pd.DataFrame({"v3": faixa_de(p3).value_counts(), "v4": faixa_de(p4).value_counts()}).reindex(ordem))

alerta_v3 = p3 >= config.FAIXA_ALERTA
print(f"\nos {alerta_v3.sum()} que a v3 poe em alerta em julho, segundo a v4:")
print(faixa_de(p4[alerta_v3]).value_counts().reindex(ordem).to_string())

os 581 do alerta de abril


    probabilidade media  marcados como 'vai sair'
v3                0.615                       581
v4                0.427                       118

a lista de julho
          v3    v4
alerta   580    59
atencao  400   171
ok       420  1170

os 580 que a v3 poe em alerta em julho, segundo a v4:
alerta      45
atencao     74
ok         461


**A conclusão:**

> **A v4 aprendeu que quem está na faixa de alerta não pede mais para sair.**

Para a v3, os 581 de abril tinham 62% de chance de sair; para a v4, 43%. Em julho, das 580 pessoas que a v3 põe em alerta, a v4 manda **461 para a faixa ok**.

Mas isso não é verdade sobre essas pessoas — é verdade sobre o **reajuste**. Elas ficaram porque receberam 8% a mais. Em julho ninguém recebeu nada ainda. Se a v4 fosse usada, o RH daria reajuste a 59 pessoas; as outras ~520 com o mesmo perfil ficariam sem nada e voltariam a sair, como em março (57%).

O retreino não aprendeu sobre o mundo. **Aprendeu o efeito da nossa própria ação** — e passaria a desfazê-la.

---
# Parte 3 — As perguntas (a entrega)

1. Na Parte 1, a tabela da 1.3 sugeria “o conceito de `mov_sal` mudou”. Depois da 2.2, qual é a explicação mais simples para a inversão? Por que ela só apareceu quando alguém perguntou ao RH o que foi feito com a lista?
2. Se a v4 tivesse sido promovida, o que aconteceria em julho com as ~520 pessoas que ela tirou do alerta? E o que o rótulo de julho (em outubro) ensinaria ao retreino seguinte?
3. Por que a v3 — o modelo “que errou” — continua sendo a melhor descrição de quem sai **sem** ação?
4. Se o RH vai continuar dando reajuste para quem o modelo aponta, qual é a pergunta que o modelo deveria responder agora? Que dado ainda não existe para responder?